# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianNeuralNetwork, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[-0.22433834 -0.22365858 -0.2835082  -0.29793126  0.25580323]
 [ 0.76748761 -0.22103116  0.63877727 -0.98139518 -0.98630846]
 [ 0.13838878 -0.71603762 -0.95031313 -0.75037699 -0.63113871]
 [-0.42870636  0.50004741 -0.73614961  0.33926435 -0.77094823]
 [ 0.53966119 -0.58122914  0.91029311  0.20258489 -0.02271797]
 [ 0.84073694  0.77694781 -0.01221466 -0.58868073 -0.4466327 ]
 [ 0.15652573 -0.51896699 -0.66508854 -0.70899065 -0.96447225]
 [-0.35206099 -0.63263094  0.49632574 -0.71362919  0.38661999]
 [ 0.29862494 -0.03564741 -0.82219736  0.8816967  -0.01040833]
 [-0.42717359  0.30540401  0.61635394 -0.45123427 -0.13729852]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_method = "VI"
update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a2', 'a2', 'a1', 'a1', 'a2', 'a2', 'a2', 'a1', 'a1', 'a1']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [1, 1, 0, 0, 0, 1, 1, 0, 1, 0]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/34 [00:00<?, ?it/s]

SVI:   3%|▎         | 1/34 [00:01<00:38,  1.18s/it]

SVI:   3%|▎         | 1/34 [00:01<00:38,  1.18s/it, loss=2718.9082]

SVI:   6%|▌         | 2/34 [00:01<00:37,  1.18s/it, loss=2336.0862]

SVI:   9%|▉         | 3/34 [00:01<00:36,  1.18s/it, loss=2902.0867]

...

SVI:  91%|█████████ | 31/34 [00:01<00:02,  1.03it/s, loss=1857.7406]

SVI:  94%|█████████▍| 32/34 [00:01<00:01,  1.03it/s, loss=2250.5332]

SVI:  97%|█████████▋| 33/34 [00:01<00:00,  1.03it/s, loss=2657.8826]

SVI: 100%|██████████| 34/34 [00:01<00:00, 21.64it/s, loss=2657.8826]

SVI: 100%|██████████| 34/34 [00:01<00:00, 21.64it/s, loss=2087.7793]